# Restoration of original segmentation: legacy vs. fixed (untokenized vs. SPM)

This notebook compares how well `mweralign` restores the *original* per-segment
segmentation of WMT24 system outputs under three conditions:

| variant       | build  | segmenter   | description                                   |
|---------------|--------|-------------|-----------------------------------------------|
| `legacy_none` | pre-fix | none (whitespace) | unconditional penalty (paper behavior)   |
| `none`        | fixed  | none (whitespace) | penalty gated on tokenization (the fix)  |
| `spm`         | fixed  | flores200 SPM     | tokenized alignment                       |

Two metrics, aggregated over systems that succeeded under **all three** variants
(so the comparison is apples-to-apples):

- **boundary_acc** — % of interior segment boundaries placed exactly on the gold boundary
- **seg_exact** — % of segments restored verbatim

The per-system TSVs are produced by `restoration_rate.py`:

```bash
python -m scripts.experiments.restoration_rate --segmenter none --legacy-penalty \
    --label legacy --per-system-tsv results/restore_legacy.tsv
python -m scripts.experiments.restoration_rate --segmenter none \
    --label fixed  --per-system-tsv results/restore_fixed.tsv
python -m scripts.experiments.restoration_rate --segmenter flores200 \
    --label spm    --per-system-tsv results/restore_spm.tsv
```

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

RESULTS = Path("results")

# variant key -> (tsv filename, human label)
VARIANTS = {
    "legacy_none": ("restore_legacy.tsv", "legacy (none)"),
    "none":        ("restore_fixed.tsv",  "fixed (none)"),
    "spm":         ("restore_spm.tsv",    "fixed (spm)"),
}

In [ ]:
def load(path):
    """Load a per-system restoration TSV, keeping only successful systems."""
    df = pd.read_csv(path, sep="\t")
    df = df[df["ok"] == 1].copy()
    for col in ["n_boundaries", "boundaries_correct", "n_segments", "segments_exact"]:
        df[col] = df[col].astype(int)
    return df.set_index(["langpair", "system"])


frames = {key: load(RESULTS / fname) for key, (fname, _) in VARIANTS.items()}
for key, df in frames.items():
    print(f"{key:12s} systems ok: {len(df)}")

In [ ]:
# Intersect on (langpair, system) pairs that succeeded under ALL variants.
common = set.intersection(*(set(df.index) for df in frames.values()))
common = sorted(common)
print(f"systems common to all variants: {len(common)}")

frames = {key: df.loc[df.index.isin(common)] for key, df in frames.items()}

In [ ]:
def aggregate(df):
    """Per-langpair boundary_acc and seg_exact (micro-averaged over systems)."""
    g = df.groupby(level="langpair").sum(numeric_only=True)
    out = pd.DataFrame(index=g.index)
    out["n_sys"] = df.groupby(level="langpair").size()
    out["boundary_acc"] = 100.0 * g["boundaries_correct"] / g["n_boundaries"]
    out["seg_exact"] = 100.0 * g["segments_exact"] / g["n_segments"]
    return out


per_lp = {key: aggregate(df) for key, df in frames.items()}

# Wide table: one row per langpair, boundary_acc + seg_exact per variant.
langpairs = sorted(per_lp["none"].index)
wide = pd.DataFrame(index=langpairs)
wide.index.name = "langpair"
wide["n_sys"] = per_lp["none"].loc[langpairs, "n_sys"]
for key, (_, label) in VARIANTS.items():
    wide[f"boundary_acc[{key}]"] = per_lp[key].loc[langpairs, "boundary_acc"]
for key, (_, label) in VARIANTS.items():
    wide[f"seg_exact[{key}]"] = per_lp[key].loc[langpairs, "seg_exact"]

# Add an ALL (micro-average over every system in every langpair) row.
all_row = {"n_sys": int(sum(len(df) for df in frames.values()) / len(frames))}
for key, df in frames.items():
    s = df.sum(numeric_only=True)
    all_row[f"boundary_acc[{key}]"] = 100.0 * s["boundaries_correct"] / s["n_boundaries"]
    all_row[f"seg_exact[{key}]"] = 100.0 * s["segments_exact"] / s["n_segments"]
wide.loc["ALL"] = all_row

wide.round(2)

## Combined TSVs

A **wide** TSV (one row per langpair, easy to read) and a **tidy/long** TSV
(`langpair, variant, metric, value` — easy to plot with any tool) are written to
`results/`.

In [ ]:
wide.round(4).to_csv(RESULTS / "restoration_comparison_wide.tsv", sep="\t")

# Tidy/long form for plotting tools.
records = []
for key, (_, label) in VARIANTS.items():
    for lp in langpairs:
        for metric in ("boundary_acc", "seg_exact"):
            records.append({
                "langpair": lp,
                "variant": key,
                "variant_label": label,
                "metric": metric,
                "value": float(per_lp[key].loc[lp, metric]),
            })
tidy = pd.DataFrame.from_records(records)
tidy.to_csv(RESULTS / "restoration_comparison_tidy.tsv", sep="\t", index=False)
print("wrote results/restoration_comparison_wide.tsv")
print("wrote results/restoration_comparison_tidy.tsv")
tidy.head()

In [ ]:
def grouped_bar(metric, title, ax):
    pivot = tidy[tidy["metric"] == metric].pivot(
        index="langpair", columns="variant", values="value")
    pivot = pivot.loc[langpairs, list(VARIANTS)]  # consistent order
    pivot.columns = [VARIANTS[c][1] for c in pivot.columns]
    pivot.plot(kind="bar", ax=ax, width=0.78)
    ax.set_title(title)
    ax.set_ylabel("% restored")
    ax.set_xlabel("")
    ax.set_ylim(0, 100)
    ax.legend(title="", fontsize=8)
    ax.grid(axis="y", alpha=0.3)


fig, axes = plt.subplots(2, 1, figsize=(12, 9))
grouped_bar("boundary_acc", "Boundary accuracy (interior boundaries on gold)", axes[0])
grouped_bar("seg_exact", "Exact segment restoration", axes[1])
fig.tight_layout()
fig.savefig(RESULTS / "restoration_comparison.png", dpi=150, bbox_inches="tight")
print("wrote results/restoration_comparison.png")
plt.show()

## Cost profile: wall time vs. problem size

`restoration_rate.py` also records, for **every** (system, segmenter) pair, the
size drivers and the measured subprocess wall time:

- `n_segments`, `total_words`, `avg_words_per_seg` — system output size
- `n_domains` — number of domain-merged groups (the DP is run per group)
- `max_domain_words` — largest merged group (whitespace words); the main driver
- `sum_domain_sq` — Σ over domains of `merged_words²`, a proxy for the O(J·S) DP work
- `wall_seconds` — wall-clock time for the full realignment subprocess

The merged group (not the individual ~21-word segment) is what makes a single
realignment take seconds: each domain's segments are concatenated into one
stream of thousands of tokens before the dynamic program runs.

In [ ]:
# Combine the per-system profiling rows from all three runs into one TSV,
# tagging each with its variant label.
prof_parts = []
for key, (fname, label) in VARIANTS.items():
    p = pd.read_csv(RESULTS / fname, sep="\t")
    p["variant"] = key
    p["variant_label"] = label
    prof_parts.append(p)
prof = pd.concat(prof_parts, ignore_index=True)

prof_cols = ["variant", "variant_label", "langpair", "system", "segmenter", "ok",
             "n_segments", "total_words", "avg_words_per_seg", "n_domains",
             "max_domain_words", "sum_domain_sq", "wall_seconds"]
prof = prof[prof_cols]
prof.to_csv(RESULTS / "realign_profile.tsv", sep="\t", index=False)
print(f"wrote results/realign_profile.tsv  ({len(prof)} rows)")

prof.groupby("variant_label")[["avg_words_per_seg", "max_domain_words",
                               "sum_domain_sq", "wall_seconds"]].mean().round(2)

In [ ]:
# Wall time scales with sum_domain_sq (the O(J·S) DP work proxy).
fig, ax = plt.subplots(figsize=(9, 6))
for key, (_, label) in VARIANTS.items():
    sub = prof[prof["variant"] == key]
    ax.scatter(sub["sum_domain_sq"], sub["wall_seconds"], s=12, alpha=0.5, label=label)
ax.set_xlabel("sum_domain_sq  (Σ merged_words² over domains)")
ax.set_ylabel("wall_seconds")
ax.set_title("Realignment cost is ~quadratic in merged-group size")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(RESULTS / "realign_cost_scatter.png", dpi=150, bbox_inches="tight")
plt.show()